### data ingestion | null handling | redundancy removal | data validation | cleansed df

In [0]:
from pyspark.sql.functions import col, count, when, coalesce
from pyspark.sql import functions as sf

In [0]:
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/mnt/bronze_layer/complete_raw_data/master_data.csv")

In [0]:
print((df.count(), len(df.columns))) # row x cols

In [0]:
df.printSchema()

In [0]:
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

In [0]:
df = df.drop('Overall motivation')

In [0]:
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

In [0]:
df = df.withColumn(
    'Fullname',
    sf.concat(
        sf.coalesce(sf.col('Firstname'), sf.lit('')),
        sf.lit(' '),
        sf.coalesce(sf.col('Surname'), sf.lit(''))
    )
)

In [0]:
rename_dict = {
	'Born country': 'born_country',
	'Born country code': 'born_country_code',
	'Born city': 'born_city',
	'Died country': 'died_country',
	'Died country code': 'died_country_code',
	'Died city': 'died_city',
    'Organization name': 'organization_name',
	'Organization city': 'organization_city',
	'Organization country': 'organization_country'
}

for old_name, new_name in rename_dict.items():
    df = df\
        .withColumnRenamed(old_name, new_name)

In [0]:
print('nulls in the respective cols')

died_cols = ['Died', 'died_country', 'died_country_code', 'died_city']

df.select(*[
    (
        sf.count(sf.when((sf.isnan(c) | sf.col(c).isNull()), c)) if t not in ("timestamp", "date")
        else sf.count(sf.when(sf.col(c).isNull(), c))
    ).alias(c)
    for c, t in df.dtypes if c in died_cols
]).show()

In [0]:
alive = 332
countries_of_dead_people = 347
country_code_dead_people = 347
cities_of_dead_people = 353

nulls_in_country_col_to_fill = nulls_in_country_code_to_fill = countries_of_dead_people - alive
cities_col_to_fill = cities_of_dead_people - alive

print(f'nulls_in_country_col_to_fill: {nulls_in_country_col_to_fill}\nnulls_in_country_code_to_fill: {nulls_in_country_code_to_fill}\ncities_col_to_fill: {cities_col_to_fill}')

In [0]:
df = df.withColumn('died_city', when(col('Died').isNotNull(), col('born_city')).otherwise(col('died_city')))
df = df.withColumn('died_country', when(col('Died').isNotNull(), col('born_country')).otherwise(col('died_country')))
df = df.withColumn('died_country', when(col('Died').isNotNull(), col('born_country')).otherwise(col('died_country')))
df = df.withColumn('died_country_code', when(col('Died').isNotNull(), col('born_country_code')).otherwise(col('died_country_code')))
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

In [0]:
df = df.withColumn('organization_name', when(col('organization_name').isNull(), 'Independent').otherwise(col('organization_name')))
df = df.withColumn('organization_city', when(col('organization_city').isNull(), col('born_city')).otherwise(col('organization_city')))
df = df.withColumn('organization_country', when(col('organization_country').isNull(), col('born_country')).otherwise(col('organization_country')))

In [0]:
df = df.filter(col('born_country').isNotNull())
df = df.filter(col('Born').isNotNull())

In [0]:
print((df.count(), len(df.columns))) # row x cols

In [0]:
null_counts = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
null_counts_row = null_counts.collect()[0].asDict()
filtered_nulls = {k: v for k, v in null_counts_row.items() if v > 0}

print(filtered_nulls)

In [0]:
df_laureates = df.select("Id", "Firstname", "Surname", "Born", "Died", "Gender", "Year", "Category", "Motivation", "Fullname").distinct()
df_countries = df.select("born_country", "born_country_code", "died_country", "died_country_code").distinct()
df_cities = df.select( "born_city", "died_city").distinct()
df_organizations = df.select("organization_name", "organization_city", "organization_country").distinct()

In [0]:
df.write.format("delta").mode("overwrite").save("/mnt/silver_layer/master_df")
df_laureates.write.format("delta").mode("overwrite").save("/mnt/silver_layer/laureates")
df_countries.write.format("delta").mode("overwrite").save("/mnt/silver_layer/countries")
df_cities.write.format("delta").mode("overwrite").save("/mnt/silver_layer/cities")
df_organizations.write.format("delta").mode("overwrite").save("/mnt/silver_layer/organizations")